In [1]:
import pandas as pd, numpy as np
import vivarium_inputs
import vivarium.gbd_mapping as gbd_mapping
from lsff_utils import paths, gbd_data
import pathlib

In [2]:
location = "India"
vehicle = "rice"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"


In [4]:
location = location.title()

In [5]:
# Simulation output lives in per-run, timestamp-named directories, so the runs
# are resolved rather than hardcoded. The child rescaling divides through by
# pregnancy outcome counts, so it reads the maternal run as well as the child one.
maternal_results = paths.latest_results(paths.MATERNAL_RESULTS_ROOT, location, vehicle)
child_results = paths.latest_results(paths.CHILD_RESULTS_ROOT, location, vehicle)
child_results

PosixPath('/mnt/share/homes/abie/projects/2026/lsff-paf-two-pass/0300_child_sim/sim_results/bouillon/nigeria/2026_09_09_22_02_21/results')

In [6]:
with gbd_data.quiet_gbd_logs():
    pop = vivarium_inputs.get_population_structure(location).value
pop[pop > 0]

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2023        2024        7.907591e+04
                  0.019178   0.076712    2023        2024        2.337381e+05
                  0.076712   0.500000    2023        2024        1.706685e+06
                  0.500000   1.000000    2023        2024        1.972513e+06
                  1.000000   2.000000    2023        2024        3.899171e+06
                  2.000000   5.000000    2023        2024        1.129739e+07
                  5.000000   10.000000   2023        2024        1.766635e+07
                  10.000000  15.000000   2023        2024        1.651034e+07
                  15.000000  20.000000   2023        2024        1.483548e+07
                  20.000000  25.000000   2023        2024        1.222554e+07
                  25.000000  30.000000   2023        2024        9.984802e+06
                  30.000000  35.000000   2023        2024        7.985380e+06
  

In [7]:
with gbd_data.quiet_gbd_logs():
    asfr = vivarium_inputs.get_measure(
        gbd_mapping.covariates.age_specific_fertility_rate, "estimate", location
    ).value
asfr[asfr > 0]

location  sex     age_start  age_end  year_start  year_end  parameter  
Nigeria   Female  10.0       15.0     2023        2024      lower_value    0.002556
                                                            mean_value     0.005714
                                                            upper_value    0.011863
                  15.0       20.0     2023        2024      lower_value    0.060154
                                                            mean_value     0.070244
                                                            upper_value    0.082133
                  20.0       25.0     2023        2024      lower_value    0.171875
                                                            mean_value     0.194484
                                                            upper_value    0.219213
                  25.0       30.0     2023        2024      lower_value    0.195823
                                                            mean_value     0.208935
    

In [8]:
asfr = asfr[asfr.index.get_level_values("parameter") == "mean_value"].droplevel(
    "parameter"
)
asfr

location  sex     age_start  age_end     year_start  year_end
Nigeria   Female  0.000000   0.019178    2023        2024        0.000000
                  0.019178   0.076712    2023        2024        0.000000
                  0.076712   0.500000    2023        2024        0.000000
                  0.500000   1.000000    2023        2024        0.000000
                  1.000000   2.000000    2023        2024        0.000000
                  2.000000   5.000000    2023        2024        0.000000
                  5.000000   10.000000   2023        2024        0.000000
                  10.000000  15.000000   2023        2024        0.005714
                  15.000000  20.000000   2023        2024        0.070244
                  20.000000  25.000000   2023        2024        0.194484
                  25.000000  30.000000   2023        2024        0.208935
                  30.000000  35.000000   2023        2024        0.194076
                  35.000000  40.000000   2023     

In [9]:
births = pop * asfr
births[births > 0]

location  sex     age_start  age_end  year_start  year_end
Nigeria   Female  10.0       15.0     2023        2024        9.434157e+04
                  15.0       20.0     2023        2024        1.042096e+06
                  20.0       25.0     2023        2024        2.377674e+06
                  25.0       30.0     2023        2024        2.086170e+06
                  30.0       35.0     2023        2024        1.549767e+06
                  35.0       40.0     2023        2024        8.335776e+05
                  40.0       45.0     2023        2024        3.613521e+05
                  45.0       50.0     2023        2024        1.408638e+05
                  50.0       55.0     2023        2024        1.094531e+04
Name: value, dtype: float64

In [10]:
births = births.sum()
f"{int(births):,}"

'8,496,788'

In [11]:
sim_baseline_births = pd.read_parquet(
    paths.measure_path(maternal_results, "pregnancy_outcome_count")
)
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
0,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,1,baseline,0,141,78.0
1,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,2,baseline,0,141,80.0
2,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,3,baseline,0,141,88.0
3,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,4,baseline,0,141,74.0
4,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,partial_term,10_to_14,invalid,5,baseline,0,141,64.0
...,...,...,...,...,...,...,...,...,...,...,...
809995,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,1,zero,0,129,0.0
809996,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,2,zero,0,129,0.0
809997,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,3,zero,0,129,0.0
809998,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,stillbirth,95_plus,severe,4,zero,0,129,0.0


In [12]:
sim_baseline_births = sim_baseline_births[
    (sim_baseline_births.scenario == "baseline")
    & (sim_baseline_births.sub_entity == "live_birth")
]
sim_baseline_births

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,input_draw,random_seed,value
5,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,1,baseline,0,141,68.0
6,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,2,baseline,0,141,61.0
7,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,3,baseline,0,141,50.0
8,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,4,baseline,0,141,49.0
9,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,10_to_14,invalid,5,baseline,0,141,51.0
...,...,...,...,...,...,...,...,...,...,...,...
807290,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,1,baseline,0,54,0.0
807291,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,2,baseline,0,54,0.0
807292,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,3,baseline,0,54,0.0
807293,pregnancy_outcome_count,custom_fertility,pregnancy_countcome,live_birth,95_plus,severe,4,baseline,0,54,0.0


In [13]:
sim_baseline_births = sim_baseline_births.groupby("input_draw").value.sum().mean()
sim_baseline_births

5215213.0

In [14]:
births

8496788.296585033

In [15]:
scalar = births / sim_baseline_births
scalar

1.629231307826743

In [16]:
# NOTE: 'person_time' was renamed to 'person_time_population' when PersonTimeObserver
# became a PublicHealthObserver, so the automated V&V loader can discover it.
for result in ["ylds", "ylls", "deaths", "person_time_population"]:
    df = pd.read_parquet(
        paths.measure_path(child_results, result)
    )
    df.value *= scalar
    path = pathlib.Path(
        f"../results/rescaled_child_results/{vehicle}/{location.lower()}/{result}.parquet"
    )
    path.parent.mkdir(exist_ok=True, parents=True)
    df.to_parquet(path)